In [6]:
from pyspark.sql.functions import col, to_timestamp, when, regexp_replace
from pyspark.sql import SparkSession
import pandas as pd
import time
import warnings

warnings.filterwarnings('ignore')

In [7]:
start_time = time.time()
df = pd.read_csv('../Data/Crime_Records.csv')
crime_records = df[['Incident ID', 'Incident Description', 'Incident Datetime', 'Incident Day of Week', 'Incident Category', 'Incident Subcategory', 'Report Datetime', 'Report Type Code','Report Type Description', 'Police District', 'Latitude', 'Longitude', 'Resolution']]
   
# some transformations
crime_records['Incident Datetime'] = pd.to_datetime(crime_records['Incident Datetime'])
crime_records['Report Datetime'] = pd.to_datetime(crime_records['Report Datetime'])
crime_records['Incident Category'] = crime_records['Incident Category'].apply(str)
crime_records['Incident Subcategory'] = crime_records['Incident Subcategory'].apply(str)
crime_records['Incident Description'] = crime_records['Incident Description'].apply(lambda d: d.replace(',', '-') )
crime_records['Incident Category'] = crime_records['Incident Category'].apply(lambda d: d.replace(',', '-') )
crime_records['Incident Subcategory'] = crime_records['Incident Subcategory'].apply(lambda d: d.replace(',', '-') )
crime_records['Incident Category'].fillna('', inplace=True)
crime_records['Incident Subcategory'].fillna('', inplace=True)
crime_records.drop_duplicates(inplace=True)
crime_records.loc[crime_records['Incident Category']=='nan', ['Incident Subcategory', 'Incident Category']] = ''
print("--- %s seconds ---" % (time.time() - start_time))

--- 76.87109351158142 seconds ---


In [8]:
spark = SparkSession.builder.appName('pyspark_tut').getOrCreate()

In [9]:
spark

In [10]:
start_time = time.time()
df_pyspark = spark.read.csv('../Data/Crime_Records.csv', header=True, inferSchema=True)
df_pyspark = df_pyspark.select('Incident ID', 'Incident Description', 'Incident Datetime', 'Incident Day of Week', 'Incident Category', 'Incident Subcategory', 'Report Datetime', 'Report Type Code', 'Report Type Description', 'Police District', 'Latitude', 'Longitude', 'Resolution')

# Convert datetime columns
df_pyspark = df_pyspark.withColumn("Incident Datetime", to_timestamp(col("Incident Datetime")))
df_pyspark = df_pyspark.withColumn("Report Datetime", to_timestamp(col("Report Datetime")))

# Ensure the categories are treated as string and handle commas in descriptions
df_pyspark = df_pyspark.withColumn("Incident Category", col("Incident Category").cast("string"))
df_pyspark = df_pyspark.withColumn("Incident Subcategory", col("Incident Subcategory").cast("string"))
df_pyspark = df_pyspark.withColumn("Incident Description", regexp_replace(col("Incident Description"), ",", "-"))
df_pyspark = df_pyspark.withColumn("Incident Category", regexp_replace(col("Incident Category"), ",", "-"))
df_pyspark = df_pyspark.withColumn("Incident Subcategory", regexp_replace(col("Incident Subcategory"), ",", "-"))

# Handle null values in categories and subcategories
df_pyspark = df_pyspark.fillna({'Incident Category': '', 'Incident Subcategory': ''})

# Drop duplicates
df_pyspark = df_pyspark.dropDuplicates()

# Replace 'nan' in 'Incident Category' with empty string and apply the same to 'Incident Subcategory'
df_pyspark = df_pyspark.withColumn(
    "Incident Category",
    when(col("Incident Category") == "nan", '').otherwise(col("Incident Category"))
)
df_pyspark = df_pyspark.withColumn(
    "Incident Subcategory",
    when(col("Incident Category") == '', '').otherwise(col("Incident Subcategory"))
)

print("--- %s seconds ---" % (time.time() - start_time))

--- 0.7365376949310303 seconds ---


In [11]:
df_pyspark.show(10)

+-----------+--------------------+-----------------+--------------------+------------------+--------------------+---------------+----------------+-----------------------+---------------+------------------+-------------------+--------------+
|Incident ID|Incident Description|Incident Datetime|Incident Day of Week| Incident Category|Incident Subcategory|Report Datetime|Report Type Code|Report Type Description|Police District|          Latitude|          Longitude|    Resolution|
+-----------+--------------------+-----------------+--------------------+------------------+--------------------+---------------+----------------+-----------------------+---------------+------------------+-------------------+--------------+
|    1049809|   Terrorist Threats|             null|              Monday|Disorderly Conduct|        Intimidation|           null|              II|                Initial|     Tenderloin| 37.78321431177312|-122.41076482950653|Open or Active|
|    1068216|       Lost Property|  